In [1]:
import numpy as np

from cal_obj_typed_py import cal_obj_typed, build_default_type_info, build_distance_matrix
from ev_typed_nn_seed_adapter import build_ev_typed_problem_context
from get_mofcn import set_ev_context, clear_ev_context
from mogabka_seeded import MOGABKA
from nn_seed_injection import make_pcc_seed_builder, SeedDecodeConfig


In [2]:

def main():
    # ========= 1. 读取数据 =========
    demand_points_info = np.loadtxt("demand_points_info.csv", delimiter=",")
    charge_points_info = np.loadtxt("charge_points_info.csv", delimiter=",")
    x_raw_flat = np.loadtxt("x_one_cell_flat.csv", delimiter=",").astype(np.float32).reshape(-1)

    # ========= 2. 参数 =========
    parameter = np.array([0.5, 10, 60, 1000, 5], dtype=float)

    # 这里你可以直接用默认 typeInfo，也可以自己写死
    typeInfo = build_default_type_info(
        parameter=parameter,
        nmax=np.array([20, 20, 10], dtype=int)   # 改成你的真实上界
    )

    M = charge_points_info.shape[0]

    # 预先缓存距离矩阵，加快 cal_obj_typed
    dist_matrix = build_distance_matrix(
        demand_points_info[:, 0:2],
        charge_points_info[:, 0:2],
    )

    # ========= 3. 构造 problem_context =========
    problem_context = build_ev_typed_problem_context(
        x_raw_flat=x_raw_flat,
        charge_points_num=M,
        cal_obj_fn=cal_obj_typed,
        demand_points_info=demand_points_info,
        charge_points_info=charge_points_info,
        parameter=parameter,
        typeInfo=typeInfo,
        type_prob=(0.50, 0.35, 0.15),
        fallback_index=0,
        stochastic_split=False,
        dist_matrix=dist_matrix,
    )

    set_ev_context(problem_context)

    # ========= 4. 构造网络种子配置 =========
    decode_cfg = SeedDecodeConfig(
        num_seeds=2,
        count_radius=1,
        top_margin=2,
        stochastic_ratio=0.25,
        priority_temperature=1.0,
        capacity_temperature=1.0,
        min_k=1,
        max_k=100,
    )

    seed_builder = make_pcc_seed_builder(
        ckpt_path="set_transformer_pcc_ckpt.pt",
        decode_cfg=decode_cfg,
    )   

    seed_cfg = {
        "enabled": True,
        "init_enabled": True,
        "init_nn_seed_count": 2,
        "builder_fn": seed_builder,
        "reinject_enabled": False,
        "reinject_generations": (20, 60),
        "reinject_count": 0,
    }

    # ========= 5. typed 编码边界 =========
    ub_vec = np.concatenate([
        np.full(M, typeInfo.nmax[0], dtype=float),
        np.full(M, typeInfo.nmax[1], dtype=float),
        np.full(M, typeInfo.nmax[2], dtype=float),
    ])

    # ========= 6. 跑算法 =========
    try:
        Fitness, POP, turePF, Result = MOGABKA(
            Max_iter=3,
            SearchAgents_no=10,
            FUN="EV_TYPED_CS",
            dim=3 * M,
            numObj=2,
            lb=0,
            ub=ub_vec,
            seed=42,
            seed_injection_config=seed_cfg,
            problem_context=problem_context,
        )
    finally:
        clear_ev_context()

    # ========= 7. 输出 =========
    print("Finished.")
    print("Fitness shape:", Fitness.shape)
    print("POP shape:", POP.shape)
    print("turePF shape:", turePF.shape)

    print("\nLast generation metrics:")
    for k, v in Result.items():
        print(k, v[-1])

    np.savetxt("final_fitness_typed.csv", Fitness, delimiter=",")
    np.savetxt("final_pop_typed.csv", POP, delimiter=",")
    np.savetxt("final_pf_typed.csv", turePF, delimiter=",")

In [6]:


if __name__ == "__main__":
    main()

1
2
3
Finished.
Fitness shape: (25, 2)
POP shape: (10, 300)
turePF shape: (0, 2)

Last generation metrics:
IGD nan
GD nan
HV nan
Spacing nan
Spread nan
Coverage nan


In [5]:
import inspect
import nn_seed_injection

print("module file:", nn_seed_injection.__file__)

src1 = inspect.getsource(nn_seed_injection.make_pcc_seed_builder)
print("lazy load generator_box ?", "generator_box" in src1)

src2 = inspect.getsource(nn_seed_injection.PCCSeedGenerator.generate_packed_seed_solutions)
print("has max_tries ?", "max_tries" in src2)
print("has while len(sols) < cfg.num_seeds ?", "while len(sols) < cfg.num_seeds" in src2)